# P3-4 · Hệ sinh thái Hugging Face

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q datasets pyvi

## Lý thuyết

In [ ]:
# Cần mạng: tải model từ Hugging Face (chạy trên Colab); thay <tên-model-trên-Hub> bằng model bạn đã đẩy lên
from transformers import pipeline
clf = pipeline("sentiment-analysis", model="<tên-model-trên-Hub>")
clf("Giảng viên dạy rất dễ hiểu")      # [{'label': ..., 'score': ...}]

In [ ]:
# Cần mạng: tải PhoBERT từ Hugging Face (chạy trên Colab)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/phobert-base-v2", num_labels=3,
    id2label={0: "tiêu cực", 1: "trung tính", 2: "tích cực"},
    label2id={"tiêu cực": 0, "trung tính": 1, "tích cực": 2})

In [ ]:
# Cần tokenizer PhoBERT ở ô trên (chạy trên Colab)
batch = tok(["Thầy dạy hay", "Phòng_học nóng quá , không"],
            padding=True, truncation=True, max_length=256, return_tensors="pt")
batch["input_ids"]        # (2, T): id token, câu ngắn được đệm bằng pad_token_id
batch["attention_mask"]   # (2, T): 1 = token thật, 0 = đệm

In [ ]:
from pyvi import ViTokenizer
ViTokenizer.tokenize("Phòng học nóng quá, không bật được điều hòa")

'Phòng học nóng quá , không bật được điều hòa'


In [ ]:
# Cần mạng: tải dataset và tokenizer từ Hugging Face (chạy trên Colab)
from datasets import load_dataset
ds = load_dataset("uitnlp/vietnamese_students_feedback")        # DatasetDict: train / validation / test
ds["train"][0]                                                  # một dòng dạng dict
ds["train"].features                                            # kiểu từng cột, tên nhãn

def preprocess(batch):
    return tok(batch["sentence"], truncation=True, max_length=256)
ds = ds.map(preprocess, batched=True)                           # token hóa, cache trên đĩa
ds = ds.rename_column("sentiment", "labels")                    # Trainer tìm cột tên "labels"
ds = ds.remove_columns(["topic"])                               # bỏ cột không dùng

In [ ]:
# Cần GPU và các ô trên: chạy trên Colab (T4, khoảng 10 phút)
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1_macro": f1_score(labels, preds, average="macro")}

args = TrainingArguments(
    "phobert-sentiment",
    learning_rate=2e-5, per_device_train_batch_size=32, num_train_epochs=3,
    warmup_steps=0.1, weight_decay=0.01, fp16=True,     # số < 1 nghĩa là tỉ lệ (10% số bước)
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1_macro",
    train_sampling_strategy="group_by_length", report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["validation"],
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
trainer.train()
trainer.evaluate(ds["test"])          # đo test một lần, ở cuối

In [ ]:
# Cần tài khoản Hugging Face: chạy trên Colab sau khi train
from huggingface_hub import login
login()                                   # token có quyền write, lấy ở trang Settings → Access Tokens
trainer.push_to_hub(commit_message="PhoBERT cảm xúc, macro F1 0.xx trên test")

## Bài tập 5: compute_metrics {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1_macro": f1_score(labels, preds, average="macro")}